In [7]:
import re
import json
import time
from datetime import datetime

In [8]:
QUERY_LIMITS = {
    "max_rows": 1000,
    "max_query_length": 5000,
    "max_execution_time_seconds": 10,
    "max_joins": 5
}

print("Query Governance Limits")
print("-----------------------")
print(f"Maximum rows        : {QUERY_LIMITS['max_rows']}")
print(f"Maximum query chars : {QUERY_LIMITS['max_query_length']}")
print(f"Maximum execution   : {QUERY_LIMITS['max_execution_time_seconds']} sec")
print(f"Maximum JOINs       : {QUERY_LIMITS['max_joins']}")

Query Governance Limits
-----------------------
Maximum rows        : 1000
Maximum query chars : 5000
Maximum execution   : 10 sec
Maximum JOINs       : 5


In [9]:
BLOCKED_SQL_PATTERNS = [
    r"\bDROP\s+TABLE\b",
    r"\bDROP\s+DATABASE\b",
    r"\bTRUNCATE\b",
    r"\bDELETE\s+FROM\b",
    r"\bUPDATE\s+.*\bSET\b",
    r"\bINSERT\s+INTO\b",
    r"\bALTER\s+TABLE\b",
    r"\bCREATE\s+TABLE\b",
    r"\bCREATE\s+DATABASE\b"
]

print("SQL safety rules loaded.")

SQL safety rules loaded.


In [10]:
def validate_sql(sql):
    errors = []

    if not isinstance(sql, str):
        return {
            "safe": False,
            "errors": ["SQL must be a string."]
        }

    sql_clean = sql.strip()

    if not sql_clean:
        return {
            "safe": False,
            "errors": ["SQL query is empty."]
        }

    if len(sql_clean) > QUERY_LIMITS["max_query_length"]:
        errors.append(
            f"Query exceeds {QUERY_LIMITS['max_query_length']} characters."
        )

    sql_upper = sql_clean.upper()

    for pattern in BLOCKED_SQL_PATTERNS:
        if re.search(pattern, sql_upper):
            errors.append(
                f"Blocked SQL operation detected: {pattern}"
            )

    join_count = len(re.findall(r"\bJOIN\b", sql_upper))

    if join_count > QUERY_LIMITS["max_joins"]:
        errors.append(
            f"Query contains {join_count} JOINs. "
            f"Maximum allowed is {QUERY_LIMITS['max_joins']}."
        )

    return {
        "safe": len(errors) == 0,
        "errors": errors
    }

In [11]:
test_queries = [
    "SELECT * FROM sales",
    "SELECT region, SUM(sales) FROM sales GROUP BY region",
    "DROP TABLE sales",
    "DELETE FROM sales"
]

for query in test_queries:
    result = validate_sql(query)


    
    print("\nSQL:", query)
    print("Safe:", result["safe"])

    if result["errors"]:
        print("Errors:")
        for error in result["errors"]:
            print(" -", error)


SQL: SELECT * FROM sales
Safe: True

SQL: SELECT region, SUM(sales) FROM sales GROUP BY region
Safe: True

SQL: DROP TABLE sales
Safe: False
Errors:
 - Blocked SQL operation detected: \bDROP\s+TABLE\b

SQL: DELETE FROM sales
Safe: False
Errors:
 - Blocked SQL operation detected: \bDELETE\s+FROM\b


In [12]:
def enforce_query_limit(sql, max_rows=None):
    if max_rows is None:
        max_rows = QUERY_LIMITS["max_rows"]

    sql_clean = sql.strip().rstrip(";")

    if re.search(r"\bLIMIT\s+\d+\b", sql_clean, re.IGNORECASE):
        sql_clean = re.sub(
            r"\bLIMIT\s+\d+\b",
            f"LIMIT {max_rows}",
            sql_clean,
            flags=re.IGNORECASE
        )
    else:
        sql_clean = f"{sql_clean}\nLIMIT {max_rows}"

    return sql_clean

In [13]:
query1 = "SELECT * FROM sales"

query2 = "SELECT * FROM customers LIMIT 5000"

print("Original Query 1:")
print(query1)

print("\nLimited Query 1:")
print(enforce_query_limit(query1))

print("\nOriginal Query 2:")
print(query2)

print("\nLimited Query 2:")
print(enforce_query_limit(query2))

Original Query 1:
SELECT * FROM sales

Limited Query 1:
SELECT * FROM sales
LIMIT 1000

Original Query 2:
SELECT * FROM customers LIMIT 5000

Limited Query 2:
SELECT * FROM customers LIMIT 1000


In [14]:
def govern_query(sql):
    validation = validate_sql(sql)

    if not validation["safe"]:
        return {
            "approved": False,
            "sql": None,
            "errors": validation["errors"]
        }

    limited_sql = enforce_query_limit(sql)

    return {
        "approved": True,
        "sql": limited_sql,
        "errors": []
    }

In [15]:
queries = [
    "SELECT * FROM sales",
    "SELECT region, SUM(sales) FROM sales GROUP BY region",
    "DROP TABLE sales"
]

for query in queries:
    result = govern_query(query)

    print("\nOriginal:")
    print(query)

    print("Approved:", result["approved"])

    if result["approved"]:
        print("Final SQL:")
        print(result["sql"])
    else:
        print("Blocked because:")
        for error in result["errors"]:
            print(" -", error)


Original:
SELECT * FROM sales
Approved: True
Final SQL:
SELECT * FROM sales
LIMIT 1000

Original:
SELECT region, SUM(sales) FROM sales GROUP BY region
Approved: True
Final SQL:
SELECT region, SUM(sales) FROM sales GROUP BY region
LIMIT 1000

Original:
DROP TABLE sales
Approved: False
Blocked because:
 - Blocked SQL operation detected: \bDROP\s+TABLE\b


In [16]:
def create_query_metadata(sql, api_call=None):
    return {
        "query_id": f"Q_{int(time.time() * 1000)}",
        "timestamp": datetime.now().isoformat(),
        "sql": sql,
        "api_call": api_call,
        "max_rows": QUERY_LIMITS["max_rows"],
        "status": "approved"
    }

In [17]:
def build_api_call(question, sql):
    return {
        "method": "POST",
        "endpoint": "/api/query",
        "body": {
            "question": question,
            "sql": sql,
            "max_rows": QUERY_LIMITS["max_rows"]
        }
    }

In [18]:
def prepare_agent_query(question, sql):
    governance = govern_query(sql)

    if not governance["approved"]:
        return {
            "success": False,
            "message": "Query blocked by cost governance.",
            "errors": governance["errors"],
            "sql": None,
            "api_call": None
        }

    final_sql = governance["sql"]

    api_call = build_api_call(
        question,
        final_sql
    )

    metadata = create_query_metadata(
        final_sql,
        api_call
    )

    return {
        "success": True,
        "message": "Query approved.",
        "sql": final_sql,
        "api_call": api_call,
        "metadata": metadata
    }

In [19]:
print(prepare_agent_query)

<function prepare_agent_query at 0x0000021F51780540>


In [20]:
test_result = metricmind_query(
    "What are the top regions by sales?",
    """
    SELECT
        region,
        SUM(sales) AS total_sales
    FROM sales
    GROUP BY region
    ORDER BY total_sales DESC
    """
)

print(test_result)

✅ QUERY APPROVED

Final SQL:
SELECT
        region,
        SUM(sales) AS total_sales
    FROM sales
    GROUP BY region
    ORDER BY total_sales DESC
LIMIT 1000

Query ID:
Q_1791629482630

Rows limited to:
1000
{'success': True, 'message': 'Query approved.', 'sql': 'SELECT\n        region,\n        SUM(sales) AS total_sales\n    FROM sales\n    GROUP BY region\n    ORDER BY total_sales DESC\nLIMIT 1000', 'api_call': {'method': 'POST', 'endpoint': '/api/query', 'body': {'question': 'What are the top regions by sales?', 'sql': 'SELECT\n        region,\n        SUM(sales) AS total_sales\n    FROM sales\n    GROUP BY region\n    ORDER BY total_sales DESC\nLIMIT 1000', 'max_rows': 1000}}, 'metadata': {'query_id': 'Q_1791629482630', 'timestamp': '2026-10-10T16:21:22.630622', 'sql': 'SELECT\n        region,\n        SUM(sales) AS total_sales\n    FROM sales\n    GROUP BY region\n    ORDER BY total_sales DESC\nLIMIT 1000', 'api_call': {'method': 'POST', 'endpoint': '/api/query', 'body': {'que

In [21]:
question = "What are the total sales by region?"

sql = """
SELECT
    region,
    SUM(sales) AS total_sales
FROM sales
GROUP BY region
"""

result = prepare_agent_query(question, sql)

print("Success:", result["success"])
print("\nSQL:")
print(result["sql"])

print("\nAPI CALL:")
print(json.dumps(result["api_call"], indent=2))

print("\nMETADATA:")
print(json.dumps(result["metadata"], indent=2))

Success: True

SQL:
SELECT
    region,
    SUM(sales) AS total_sales
FROM sales
GROUP BY region
LIMIT 1000

API CALL:
{
  "method": "POST",
  "endpoint": "/api/query",
  "body": {
    "question": "What are the total sales by region?",
    "sql": "SELECT\n    region,\n    SUM(sales) AS total_sales\nFROM sales\nGROUP BY region\nLIMIT 1000",
    "max_rows": 1000
  }
}

METADATA:
{
  "query_id": "Q_1791629484656",
  "timestamp": "2026-10-10T16:21:24.656800",
  "sql": "SELECT\n    region,\n    SUM(sales) AS total_sales\nFROM sales\nGROUP BY region\nLIMIT 1000",
  "api_call": {
    "method": "POST",
    "endpoint": "/api/query",
    "body": {
      "question": "What are the total sales by region?",
      "sql": "SELECT\n    region,\n    SUM(sales) AS total_sales\nFROM sales\nGROUP BY region\nLIMIT 1000",
      "max_rows": 1000
    }
  },
  "max_rows": 1000,
  "status": "approved"
}


In [22]:
expensive_sql = """
SELECT *
FROM sales
"""

result = prepare_agent_query(
    "Show me all sales records",
    expensive_sql
)

print("Success:", result["success"])
print("Final SQL:")
print(result["sql"])

Success: True
Final SQL:
SELECT *
FROM sales
LIMIT 1000


In [23]:
dangerous_sql = "DROP TABLE sales"

result = prepare_agent_query(
    "Delete the sales table",
    dangerous_sql
)

print("Success:", result["success"])

print("\nMessage:")
print(result["message"])

print("\nErrors:")
for error in result["errors"]:
    print("-", error)

Success: False

Message:
Query blocked by cost governance.

Errors:
- Blocked SQL operation detected: \bDROP\s+TABLE\b


In [24]:
def view_sql(result):
    if not result.get("success"):
        return "No SQL available because the query was blocked."

    return result.get("sql", "")

In [25]:
def view_api_call(result):
    if not result.get("success"):
        return {
            "error": "No API call was generated because the query was blocked."
        }

    return result.get("api_call", {})

In [26]:
print("===== VIEW SQL =====")
print(view_sql(result))

===== VIEW SQL =====
No SQL available because the query was blocked.


In [27]:
print("===== VIEW API CALL =====")

api_data = view_api_call(result)

print(
    json.dumps(
        api_data,
        indent=2,
        default=str
    )
)

===== VIEW API CALL =====
{
  "error": "No API call was generated because the query was blocked."
}


In [28]:
ui_state = {
    "show_sql": False,
    "show_api": False
}

print("Initial UI State:")
print(ui_state)

Initial UI State:
{'show_sql': False, 'show_api': False}


In [29]:
def toggle_sql_view():
    ui_state["show_sql"] = not ui_state["show_sql"]

    if ui_state["show_sql"]:
        print("SQL view: OPEN")
    else:
        print("SQL view: CLOSED")

In [30]:
def toggle_api_view():
    ui_state["show_api"] = not ui_state["show_api"]

    if ui_state["show_api"]:
        print("API view: OPEN")
    else:
        print("API view: CLOSED")

In [31]:
print("===== METRICMIND TRANSPARENCY UI =====")

print("\n[View SQL]")
toggle_sql_view()

if ui_state["show_sql"]:
    print("\nSQL:")
    print(view_sql(result))

print("\n[View API Call]")
toggle_api_view()

if ui_state["show_api"]:
    print("\nAPI Call:")
    print(
        json.dumps(
            view_api_call(result),
            indent=2,
            default=str
        )
    )

===== METRICMIND TRANSPARENCY UI =====

[View SQL]
SQL view: OPEN

SQL:
No SQL available because the query was blocked.

[View API Call]
API view: OPEN

API Call:
{
  "error": "No API call was generated because the query was blocked."
}


In [32]:
def metricmind_query(question, sql):
    result = prepare_agent_query(question, sql)

    if not result["success"]:
        print("❌ QUERY BLOCKED")
        print("\nReason:")

        for error in result["errors"]:
            print("-", error)

        return result

    print("✅ QUERY APPROVED")
    print("\nFinal SQL:")
    print(result["sql"])

    print("\nQuery ID:")
    print(result["metadata"]["query_id"])

    print("\nRows limited to:")
    print(result["metadata"]["max_rows"])

    return result

In [33]:
print(metricmind_query)

<function metricmind_query at 0x0000021F51780720>


In [34]:
print(metricmind_query)

<function metricmind_query at 0x0000021F51780720>


In [35]:
question = "What are the top regions by sales?"

sql = """
SELECT
    region,
    SUM(sales) AS total_sales
FROM sales
GROUP BY region
ORDER BY total_sales DESC
"""

final_result = metricmind_query(
    question,
    sql
)

✅ QUERY APPROVED

Final SQL:
SELECT
    region,
    SUM(sales) AS total_sales
FROM sales
GROUP BY region
ORDER BY total_sales DESC
LIMIT 1000

Query ID:
Q_1791629509613

Rows limited to:
1000


In [36]:
print("\n" + "=" * 60)
print("METRICMIND — TRANSPARENCY PANEL")
print("=" * 60)

print("\n📌 QUERY ID:")
print(final_result["metadata"]["query_id"])

print("\n📌 VIEW SQL:")
print(final_result["sql"])

print("\n📌 VIEW API CALL:")
print(
    json.dumps(
        final_result["api_call"],
        indent=2,
        default=str
    )
)

print("\n📌 GOVERNANCE:")
print("Maximum rows:", QUERY_LIMITS["max_rows"])
print("Maximum joins:", QUERY_LIMITS["max_joins"])
print("Maximum query length:", QUERY_LIMITS["max_query_length"])

print("\n" + "=" * 60)
print("Week 5 Cost Governance + Transparency Complete")
print("=" * 60)


METRICMIND — TRANSPARENCY PANEL

📌 QUERY ID:
Q_1791629509613

📌 VIEW SQL:
SELECT
    region,
    SUM(sales) AS total_sales
FROM sales
GROUP BY region
ORDER BY total_sales DESC
LIMIT 1000

📌 VIEW API CALL:
{
  "method": "POST",
  "endpoint": "/api/query",
  "body": {
    "question": "What are the top regions by sales?",
    "sql": "SELECT\n    region,\n    SUM(sales) AS total_sales\nFROM sales\nGROUP BY region\nORDER BY total_sales DESC\nLIMIT 1000",
    "max_rows": 1000
  }
}

📌 GOVERNANCE:
Maximum rows: 1000
Maximum joins: 5
Maximum query length: 5000

Week 5 Cost Governance + Transparency Complete


In [37]:
import fastapi
import uvicorn

print("FastAPI:", fastapi.__version__)
print("Uvicorn:", uvicorn.__version__)

FastAPI: 0.139.2
Uvicorn: 0.51.0


In [38]:
from fastapi import FastAPI
from pydantic import BaseModel
import uvicorn
import threading
import nest_asyncio

# Allow FastAPI/Uvicorn to run inside Jupyter
nest_asyncio.apply()

app = FastAPI(
    title="MetricMind API",
    description="Agentic Semantic BI Engine API",
    version="1.0.0"
)


# -----------------------------
# Request Model
# -----------------------------

class MetricMindRequest(BaseModel):
    question: str
    sql: str


# -----------------------------
# Health Check
# -----------------------------

@app.get("/")
def root():
    return {
        "status": "online",
        "service": "MetricMind API",
        "version": "1.0.0"
    }


# -----------------------------
# MetricMind Query API
# -----------------------------

@app.post("/query")
def query_metricmind(request: MetricMindRequest):

    result = metricmind_query(
        request.question,
        request.sql
    )

    return result


# -----------------------------
# Start API Server
# -----------------------------

def start_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8001
    )


server_thread = threading.Thread(
    target=start_server,
    daemon=True
)

server_thread.start()

print("=" * 60)
print("🚀 METRICMIND API STARTED")
print("=" * 60)
print("Local API: http://127.0.0.1:8001")
print("Swagger UI: http://127.0.0.1:8001/docs")
print("OpenAPI: http://127.0.0.1:8001/openapi.json")
print("=" * 60)

🚀 METRICMIND API STARTED
Local API: http://127.0.0.1:8001
Swagger UI: http://127.0.0.1:8001/docs
OpenAPI: http://127.0.0.1:8001/openapi.json


INFO:     Started server process [7708]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 10048] error while attempting to bind on address ('0.0.0.0', 8001): [winerror 10048] only one usage of each socket address (protocol/network address/port) is normally permitted
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


In [39]:
test_result = metricmind_query(
    "What are the top regions by sales?",
    """
    SELECT
        region,
        SUM(sales) AS total_sales
    FROM sales
    GROUP BY region
    ORDER BY total_sales DESC
    """
)

print(test_result)

✅ QUERY APPROVED

Final SQL:
SELECT
        region,
        SUM(sales) AS total_sales
    FROM sales
    GROUP BY region
    ORDER BY total_sales DESC
LIMIT 1000

Query ID:
Q_1791629516869

Rows limited to:
1000
{'success': True, 'message': 'Query approved.', 'sql': 'SELECT\n        region,\n        SUM(sales) AS total_sales\n    FROM sales\n    GROUP BY region\n    ORDER BY total_sales DESC\nLIMIT 1000', 'api_call': {'method': 'POST', 'endpoint': '/api/query', 'body': {'question': 'What are the top regions by sales?', 'sql': 'SELECT\n        region,\n        SUM(sales) AS total_sales\n    FROM sales\n    GROUP BY region\n    ORDER BY total_sales DESC\nLIMIT 1000', 'max_rows': 1000}}, 'metadata': {'query_id': 'Q_1791629516869', 'timestamp': '2026-10-10T16:21:56.869852', 'sql': 'SELECT\n        region,\n        SUM(sales) AS total_sales\n    FROM sales\n    GROUP BY region\n    ORDER BY total_sales DESC\nLIMIT 1000', 'api_call': {'method': 'POST', 'endpoint': '/api/query', 'body': {'que

In [40]:
print("Checking available data objects...")

for name, obj in list(globals().items()):
    if not name.startswith("_"):
        if hasattr(obj, "columns") and hasattr(obj, "head"):
            print(f"\nDataFrame found: {name}")
            print("Columns:", list(obj.columns))
            print(obj.head(3))

Checking available data objects...


In [41]:
import pandas as pd

# Clearly labelled demonstration data
sales_df = pd.DataFrame({
    "quarter": ["Q1", "Q2", "Q3", "Q4"],
    "region": ["Europe", "Europe", "Europe", "Europe"],
    "revenue": [100000, 120000, 135000, 150000],
    "cost": [70000, 82000, 90000, 98000]
})

print("Sample demonstration data:")
display(sales_df)

Sample demonstration data:


,quarter,region,revenue,cost
0,Q1,Europe,100000,70000
1,Q2,Europe,120000,82000
2,Q3,Europe,135000,90000
3,Q4,Europe,150000,98000


In [42]:

def metricmind_query(question, sql=""):
    """
    MetricMind demo query handler.
    Uses clearly labelled sample data, not real company data.
    """

    question_lower = question.lower().strip()

    # Resolve a supported natural-language question
    if "revenue" in question_lower and "q3" in question_lower:
        selected = sales_df[sales_df["quarter"] == "Q3"]
        answer = int(selected["revenue"].sum())

        return {
            "success": True,
            "message": "Answer calculated using sample demonstration data.",
            "answer": f"Q3 sample revenue: {answer:,}",
            "value": answer,
            "data_source": "Sample demonstration data",
            "sql": (
                "SELECT SUM(revenue) AS q3_revenue "
                "FROM sales WHERE quarter = 'Q3' LIMIT 1000"
            ),
            "metadata": {
                "status": "approved_demo",
                "data_source": "Sample demonstration data"
            }
        }

    return {
        "success": False,
        "message": (
            "This demo currently supports Q3 revenue questions only."
        ),
        "answer": None,
        "sql": None,
        "metadata": {"status": "unsupported"}
    }


# Test the function before connecting it to Retool
result = metricmind_query("What was our Q3 revenue?")
print(result)


{'success': True, 'message': 'Answer calculated using sample demonstration data.', 'answer': 'Q3 sample revenue: 135,000', 'value': 135000, 'data_source': 'Sample demonstration data', 'sql': "SELECT SUM(revenue) AS q3_revenue FROM sales WHERE quarter = 'Q3' LIMIT 1000", 'metadata': {'status': 'approved_demo', 'data_source': 'Sample demonstration data'}}
INFO:     127.0.0.1:63849 - "POST /query HTTP/1.1" 200 OK
INFO:     127.0.0.1:55461 - "POST /query HTTP/1.1" 200 OK
INFO:     35.90.103.133:0 - "POST /query HTTP/1.1" 200 OK
INFO:     35.90.103.133:0 - "POST /query HTTP/1.1" 200 OK


In [43]:

def metricmind_query(question, sql=""):
    """MetricMind demo using sample data only."""

    q = question.lower()

    # Identify the requested quarter
    quarter = next(
        (f"Q{i}" for i in range(1, 5) if f"q{i}" in q),
        None
    )

    # Support revenue questions for Q1–Q4
    if quarter and "revenue" in q:
        selected = sales_df[sales_df["quarter"] == quarter]

        if selected.empty:
            return {
                "success": False,
                "message": f"No sample data available for {quarter}.",
                "answer": None,
                "metadata": {"status": "unsupported"}
            }

        value = int(selected["revenue"].sum())
        sql_text = (
            f"SELECT SUM(revenue) AS {quarter.lower()}_revenue "
            f"FROM sales WHERE quarter = '{quarter}' LIMIT 1000"
        )

        return {
            "success": True,
            "message": "Calculated from sample demonstration data.",
            "answer": f"{quarter} sample revenue: {value:,}",
            "value": value,
            "data_source": "Sample demonstration data",
            "sql": sql_text,
            "metadata": {
                "status": "approved_demo",
                "data_source": "Sample demonstration data"
            }
        }

    return {
        "success": False,
        "message": (
            "This demo supports Q1, Q2, Q3, and Q4 revenue questions."
        ),
        "answer": None,
        "sql": None,
        "metadata": {"status": "unsupported"}
    }


print(metricmind_query("What was our Q2 revenue?"))


{'success': True, 'message': 'Calculated from sample demonstration data.', 'answer': 'Q2 sample revenue: 120,000', 'value': 120000, 'data_source': 'Sample demonstration data', 'sql': "SELECT SUM(revenue) AS q2_revenue FROM sales WHERE quarter = 'Q2' LIMIT 1000", 'metadata': {'status': 'approved_demo', 'data_source': 'Sample demonstration data'}}
INFO:     35.90.103.133:0 - "POST /query HTTP/1.1" 200 OK
INFO:     35.90.103.132:0 - "POST /query HTTP/1.1" 200 OK
INFO:     35.90.103.132:0 - "POST /query HTTP/1.1" 200 OK


In [44]:

def metricmind_query(question, sql=""):
    """MetricMind business metrics demo using sample data."""

    q = question.lower()

    # Find a requested quarter, if present
    quarter = next(
        (f"Q{i}" for i in range(1, 5) if f"q{i}" in q),
        None
    )

    # Revenue, profit, and margin calculations
    if any(word in q for word in ["revenue", "profit", "margin"]):

        df = sales_df.copy()
        if quarter:
            df = df[df["quarter"] == quarter]

        if df.empty:
            return {
                "success": False,
                "message": "No matching sample data found.",
                "answer": None,
                "metadata": {"status": "unsupported"}
            }

        df["profit"] = df["revenue"] - df["cost"]
        df["profit_margin"] = (
            df["profit"] / df["revenue"] * 100
        )

        # Quarter-specific revenue
        if "revenue" in q and "margin" not in q and "profit" not in q:
            value = int(df["revenue"].sum())
            metric = "revenue"
            answer = (
                f"{quarter} sample revenue: {value:,}"
                if quarter else f"Total sample revenue: {value:,}"
            )

        # Quarter-specific profit
        elif "profit" in q and "margin" not in q:
            value = int(df["profit"].sum())
            metric = "profit"
            answer = (
                f"{quarter} sample profit: {value:,}"
                if quarter else f"Total sample profit: {value:,}"
            )

        # Profit margin
        elif "margin" in q:
            value = (
                df["profit"].sum() / df["revenue"].sum() * 100
            )
            metric = "profit_margin"

            if quarter:
                answer = f"{quarter} sample profit margin: {value:.2f}%"
            else:
                lines = [
                    f"{row.quarter}: {row.profit_margin:.2f}%"
                    for row in df.itertuples()
                ]
                answer = "Sample profit margins by quarter:\n" + "\n".join(lines)

        else:
            return {
                "success": False,
                "message": "Please specify revenue, profit, or profit margin.",
                "answer": None,
                "metadata": {"status": "unsupported"}
            }

        # Display a representative governed SQL statement
        sql_text = (
            f"SELECT SUM(revenue - cost) AS {metric} "
            "FROM sales"
        )
        if quarter:
            sql_text += f" WHERE quarter = '{quarter}'"
        sql_text += " LIMIT 1000"

        return {
            "success": True,
            "message": "Calculated from sample demonstration data.",
            "answer": answer,
            "value": round(float(value), 2),
            "data_source": "Sample demonstration data",
            "sql": sql_text,
            "metadata": {
                "status": "approved_demo",
                "data_source": "Sample demonstration data"
            }
        }

    return {
        "success": False,
        "message": "Ask about revenue, profit, or profit margin.",
        "answer": None,
        "sql": None,
        "metadata": {"status": "unsupported"}
    }


# Quick test
for question in [
    "What was our Q2 revenue?",
    "What was our Q3 profit?",
    "What was our Q4 profit margin?",
    "Show profit margins by quarter"
]:
    print(question)
    print(metricmind_query(question)["answer"])
    print("-" * 40)


What was our Q2 revenue?
Q2 sample revenue: 120,000
----------------------------------------
What was our Q3 profit?
Q3 sample profit: 45,000
----------------------------------------
What was our Q4 profit margin?
Q4 sample profit margin: 34.67%
----------------------------------------
Show profit margins by quarter
Sample profit margins by quarter:
Q1: 30.00%
Q2: 31.67%
Q3: 33.33%
Q4: 34.67%
----------------------------------------
INFO:     35.90.103.133:0 - "POST /query HTTP/1.1" 200 OK
INFO:     35.90.103.132:0 - "POST /query HTTP/1.1" 200 OK
INFO:     35.90.103.132:0 - "POST /query HTTP/1.1" 200 OK
INFO:     35.90.103.133:0 - "POST /query HTTP/1.1" 200 OK
INFO:     35.90.103.132:0 - "POST /query HTTP/1.1" 200 OK


In [45]:

def metricmind_query(question, sql=""):
    """MetricMind conversational BI demo using sample data."""

    import pandas as pd

    q = question.lower().strip()
    df = sales_df.copy()

    # Validate required sample-data columns
    required = {"quarter", "region", "revenue", "cost"}
    if not required.issubset(df.columns):
        return {
            "success": False,
            "message": "Sample data is missing required columns.",
            "answer": "Unable to analyze the available data.",
            "metadata": {"status": "unsupported"}
        }

    df["profit"] = df["revenue"] - df["cost"]
    df["profit_margin"] = (
        df["profit"] / df["revenue"].replace(0, float("nan")) * 100
    )

    def respond(answer, sql_text, value=None):
        return {
            "success": True,
            "message": "Analysis calculated from sample demonstration data.",
            "answer": answer,
            "value": value,
            "data_source": "Sample demonstration data",
            "sql": sql_text,
            "metadata": {
                "status": "approved_demo",
                "data_source": "Sample demonstration data"
            }
        }

    def unsupported(message):
        return {
            "success": False,
            "message": message,
            "answer": message,
            "sql": None,
            "metadata": {"status": "unsupported"}
        }

    # Optional region filter
    if "europe" in q:
        df = df[df["region"].str.lower() == "europe"]

    if df.empty:
        return unsupported("No matching sample data was found.")

    # Quarter-specific questions
    quarter = next(
        (f"Q{i}" for i in range(1, 5) if f"q{i}" in q),
        None
    )

    # 1. Show sales/revenue over the four quarters
    if ("last four quarters" in q or "by quarter" in q
            or "quarterly sales" in q):
        grouped = (
            df.groupby("quarter", sort=True)
            .agg(revenue=("revenue", "sum"),
                 cost=("cost", "sum"),
                 profit=("profit", "sum"))
            .reset_index()
        )
        lines = [
            f'{r.quarter}: Revenue {r.revenue:,.0f}, '
            f'Cost {r.cost:,.0f}, Profit {r.profit:,.0f}'
            for r in grouped.itertuples()
        ]
        return respond(
            "Quarterly results (sample data):\n" + "\n".join(lines),
            "SELECT quarter, SUM(revenue), SUM(cost) "
            "FROM sales GROUP BY quarter LIMIT 1000"
        )

    # 2. Compare revenue and cost by region
    if "by region" in q or "compare revenue and cost" in q:
        grouped = (
            df.groupby("region")
            .agg(revenue=("revenue", "sum"),
                 cost=("cost", "sum"))
            .reset_index()
        )
        lines = [
            f"{r.region}: Revenue {r.revenue:,.0f}, "
            f"Cost {r.cost:,.0f}, "
            f"Profit {(r.revenue-r.cost):,.0f}"
            for r in grouped.itertuples()
        ]
        return respond(
            "Revenue and cost by region (sample data):\n"
            + "\n".join(lines),
            "SELECT region, SUM(revenue), SUM(cost) "
            "FROM sales GROUP BY region LIMIT 1000"
        )

    # 3. Explain whether the latest quarter's margin dropped
    if ("why" in q or "caused" in q or "drop" in q
            or "decline" in q or "last quarter" in q):
        grouped = (
            df.groupby("quarter")
            .agg(revenue=("revenue", "sum"),
                 cost=("cost", "sum"))
            .reset_index()
        )
        order = {"Q1": 1, "Q2": 2, "Q3": 3, "Q4": 4}
        grouped["_order"] = grouped["quarter"].map(order)
        grouped = grouped.sort_values("_order")

        if len(grouped) < 2:
            return unsupported(
                "At least two quarters are needed for margin comparison."
            )

        previous = grouped.iloc[-2]
        latest = grouped.iloc[-1]

        prev_margin = (
            (previous["revenue"] - previous["cost"])
            / previous["revenue"] * 100
        )
        latest_margin = (
            (latest["revenue"] - latest["cost"])
            / latest["revenue"] * 100
        )
        change = latest_margin - prev_margin

        if change < 0:
            explanation = (
                f"Margin declined from {prev_margin:.2f}% in "
                f"{previous['quarter']} to {latest_margin:.2f}% in "
                f"{latest['quarter']} ({change:.2f} percentage points). "
                f"Revenue changed by "
                f"{latest['revenue']-previous['revenue']:,.0f}, while cost "
                f"changed by {latest['cost']-previous['cost']:,.0f}. "
                "These figures show the change, but more detailed cost "
                "categories are needed to establish the underlying cause."
            )
        elif change > 0:
            explanation = (
                f"No margin decline is shown in this sample. Margin rose "
                f"from {prev_margin:.2f}% in {previous['quarter']} to "
                f"{latest_margin:.2f}% in {latest['quarter']} "
                f"(an increase of {change:.2f} percentage points). "
                f"Revenue changed by "
                f"{latest['revenue']-previous['revenue']:,.0f}, and cost "
                f"changed by {latest['cost']-previous['cost']:,.0f}. "
                "The sample does not establish a cause for any real-world "
                "margin movement."
            )
        else:
            explanation = (
                f"Margin remained unchanged at {latest_margin:.2f}% "
                f"between {previous['quarter']} and {latest['quarter']}."
            )

        return respond(
            explanation,
            "SELECT quarter, SUM(revenue), SUM(cost) "
            "FROM sales GROUP BY quarter LIMIT 1000",
            round(float(change), 2)
        )

    # 4. Revenue questions
    if "revenue" in q or "sales" in q:
        chosen = df[df["quarter"] == quarter] if quarter else df
        if chosen.empty:
            return unsupported("No sample data for that quarter.")
        value = int(chosen["revenue"].sum())
        label = f"{quarter} " if quarter else ""
        return respond(
            f"{label}sample revenue: {value:,}",
            "SELECT SUM(revenue) FROM sales"
            + (f" WHERE quarter = '{quarter}'" if quarter else "")
            + " LIMIT 1000",
            value
        )

    # 5. Profit margin questions
    if "margin" in q:
        chosen = df[df["quarter"] == quarter] if quarter else df
        revenue = chosen["revenue"].sum()
        if revenue == 0:
            return unsupported("Cannot calculate margin with zero revenue.")
        margin = (
            (revenue - chosen["cost"].sum()) / revenue * 100
        )
        label = f"{quarter} " if quarter else ""
        return respond(
            f"{label}sample profit margin: {margin:.2f}%",
            "SELECT SUM(revenue), SUM(cost) FROM sales"
            + (f" WHERE quarter = '{quarter}'" if quarter else "")
            + " LIMIT 1000",
            round(float(margin), 2)
        )

    # 6. Profit questions
    if "profit" in q:
        chosen = df[df["quarter"] == quarter] if quarter else df
        value = int((chosen["revenue"] - chosen["cost"]).sum())
        return respond(
            f"{quarter + ' ' if quarter else ''}sample profit: {value:,}",
            "SELECT SUM(revenue - cost) FROM sales LIMIT 1000",
            value
        )

    return unsupported(
        "This demo supports revenue, sales, profit, profit margin, "
        "quarter comparisons, and revenue/cost by region."
    )


print("MetricMind query function updated successfully.")


MetricMind query function updated successfully.
INFO:     35.90.103.133:0 - "POST /query HTTP/1.1" 200 OK
INFO:     35.90.103.134:0 - "POST /query HTTP/1.1" 200 OK
INFO:     35.90.103.132:0 - "POST /query HTTP/1.1" 200 OK
INFO:     35.90.103.133:0 - "POST /query HTTP/1.1" 200 OK
INFO:     35.90.103.132:0 - "POST /query HTTP/1.1" 200 OK
INFO:     35.90.103.132:0 - "POST /query HTTP/1.1" 200 OK
